# Three systems, one duty, three rungs

> **Demonstration only:** frozen synthetic data, not evidence about any real decision. A rung is how a conclusion was reached, not a confidence score.

The next cell imports each shipped system and evaluates the same duty through `check_conformance`. The table keeps the result fields that matter: verdict, rung, basis, and the evidence summary.

In [1]:
import html
from dataclasses import replace

from IPython.display import HTML, display

from reasonsmith.examples import neural_scorer, probabilistic_scorer, symbolic_rules
from reasonsmith.report import check_conformance
from reasonsmith.spec import load_pack


def show_table(rows, columns, title=None):
    heading = f"<h3>{html.escape(title)}</h3>" if title else ""
    head = "".join(f"<th>{html.escape(column)}</th>" for column in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{html.escape(str(row.get(column, '—')))}</td>" for column in columns
    ) + "</tr>" for row in rows)
    display(HTML(f"{heading}<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

pack = load_pack("ecoa")
requirement = pack.get_requirement("ecoa_reg_b_1002_9_b_2_specific_reasons")
one_duty = replace(pack, id="ecoa:specific-reasons", requirements=(requirement,))
reports = {}
for name, factory in {
    "neural": neural_scorer.system_under_test,
    "probabilistic": probabilistic_scorer.system_under_test,
    "symbolic": symbolic_rules.system_under_test,
}.items():
    reports[name] = check_conformance(factory(), one_duty)

show_table([{"System": name, "Duty": item["requirement_id"], "Verdict": item["verdict"],
             "Rung": item["strength"], "Basis": item["basis"],
             "Witness / refusal": item["evidence_summary"]}
            for name, report in reports.items()
            for item in [report.to_dict()["results"][0]]],
           ["System", "Duty", "Verdict", "Rung", "Basis", "Witness / refusal"],
           "One duty, three evidence rungs")


System,Duty,Verdict,Rung,Basis,Witness / refusal
neural,ecoa_reg_b_1002_9_b_2_specific_reasons,satisfied,observed,behavioural,"Observed over 3 decision(s): state monitor for 'present(artifact_logs_reason_explanation) -> ( present(provenance_model_version) and present(scope_statements_local_vs_global) and not contains(artifact_logs_reason_explanation, ""internal standards"") and not contains(artifact_logs_reason_explanation, ""internal policies"") and not contains(artifact_logs_reason_explanation, ""failed to achieve a qualifying score""))' satisfied at every decision step."
probabilistic,ecoa_reg_b_1002_9_b_2_specific_reasons,satisfied,probed,behavioural,"Probed: no counterexample to 'present(artifact_logs_reason_explanation) -> ( present(provenance_model_version) and present(scope_statements_local_vs_global) and not contains(artifact_logs_reason_explanation, ""internal standards"") and not contains(artifact_logs_reason_explanation, ""internal policies"") and not contains(artifact_logs_reason_explanation, ""failed to achieve a qualifying score""))' in 200 input(s) replayed through the system's own decide() (seed 0, generated by perturbing 2 recorded decision(s) over 10 field(s)). This is a bounded search, not a proof: the property is unchecked outside the inputs this budget names."
symbolic,ecoa_reg_b_1002_9_b_2_specific_reasons,satisfied,proved,behavioural,"Proved for all inputs: formal solver verified requirement 'present(artifact_logs_reason_explanation) -> ( present(provenance_model_version) and present(scope_statements_local_vs_global) and not contains(artifact_logs_reason_explanation, ""internal standards"") and not contains(artifact_logs_reason_explanation, ""internal policies"") and not contains(artifact_logs_reason_explanation, ""failed to achieve a qualifying score""))' holds across all valid inputs under system constraints. Limit of this proof: `real` is the exact rationals to the solver and IEEE-754 float64 to the system, so this holds over the rationals and not over the arithmetic the system runs. A property that depends on rounding can be proved here and still fail in execution."


`observed`, `probed`, and `proved` are distinct evidence contracts. Now ask the neural log for the shipped reason-deletion duty: it has no inference artefact, so the honest answer is an unattainable refusal with its reason, not a weaker pass.

In [2]:
artifact_requirement = pack.get_requirement(
    "ecoa_reg_b_1002_9_b_2_principal_reasons_complete"
)
artifact_duty = replace(pack, id="ecoa:principal-reasons", requirements=(artifact_requirement,))
refusal_report = check_conformance(neural_scorer.system_under_test(), artifact_duty)
refusal = refusal_report.to_dict()["results"][0]
show_table([{"Duty": refusal["requirement_id"], "Verdict": refusal["verdict"],
             "Outcome": refusal["outcome"], "Rung": refusal["strength"],
             "Basis": refusal["basis"], "Witness / refusal": refusal["evidence_summary"]}],
           ["Duty", "Verdict", "Outcome", "Rung", "Basis", "Witness / refusal"],
           "Honest refusal")

Duty,Verdict,Outcome,Rung,Basis,Witness / refusal
ecoa_reg_b_1002_9_b_2_principal_reasons_complete,inconclusive,unattainable,unattainable,artifact,"Unattainable as built: the system declares no capability to emit artifact_logs_deleted_reason_count, so no amount of testing can discharge this requirement. Determined from declared capabilities alone; the system was not executed."


The refusal is a usable result: the log-only system cannot expose the artefact needed by this duty, and no amount of replay or extra log rows changes that. See [`docs/theory/08-evidence.md`](../docs/theory/08-evidence.md) for the authoritative ladder and refusal semantics.